# Extract and clean

The goal of this notebook is to EXTRACT and CLEAN all power data from the four readable prize systems, both metered and inverter.

This notebook will only need to be run once.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from extract_and_clean import Clean


system_ids = [2105, 2107, 7333, 9068] #the relevant ids

def systemPath(system_id):
    return f'../../../../data_ds_project/systems/prize/{system_id}/'

We will create an intermediate dictionary to store all information about data locations.

# Structure of the dictionary

- Keys: system id's
- Values: tuple (meter_{system_id}, inverter_{system_id})

## Structure of the values

### meters

Each meter_{system_id} is a **list** of tuples. The list is structured as follows:

> [(file name, time column, power column),
 <br>
 ...]

The data extrated from all tuples should be CONCATENATED, as it should belong to the same meter. This happens if data is stretched across several files.

### inverters

Each inverter_{system_id} is a **list**. The list is structured as follows:

> [ [inverter 1 information],
<br>
[inverter 2 information],
<br>
...]

[inverter *n* information] is a **list**. It is a list of tuples, structured as follows:

> [(file name, time column, power column),
 <br>
 ...]

The data extrated from all tuples should be CONCATENATED, as it should belong to the same inverter.

In [2]:
#information about accessing the data
#only aggregate meters will be considered
meter_2105 = [('2105_meter_data.csv', 'measured_on', 'meter_ac_output_(kwatts)_meter_150161')]
inverter_2105 =[] #only has data in kWh, so power data would have to be approximated.

meter_2107 = [] #cannot currently find.
#build the inverter list. There are a LOT of columns of inverter data.
inverter_2107 = []
df = pd.read_csv(systemPath(2107)+'2107_electrical_data_v1.csv') #won't need this again in the short term
cols = list(df.columns) #get column names so we can pick out the relevant columns -- those that contain "_ac_power_inv_"
relevant_columns =[] #each entry will contain the column name corresponding to the data of an inverter
for col in cols:
    if "_ac_power_inv_" in col:
        relevant_columns.append(col)
for col in relevant_columns: #now we append to the inverter list
    inverter_2107.append([('2107_electrical_data_v1.csv', "measured_on", col)])

meter_7333 = [] #no data
inverter_7333 = []   #utc_measured_on
#will assume all the files have the same column names
df = pd.read_csv(systemPath(7333)+'7333_5_min_ac_20160101_20161231.csv')
cols = list(df.columns) #get column names so we can pick out the relevant columns -- those that contain "_ac_power_inv_"
relevant_columns =[] #each entry will contain the column name corresponding to the data of an inverter
for col in cols:
    if "power" in col:
        relevant_columns.append(col)
folder_path = Path(systemPath(7333))
for col in relevant_columns: #now we append to the inverter list
    #have to loop through all the file names
    invlist = []
    for file in folder_path.glob('*_ac_*'):
        invlist.append((file.name, 'utc_measured_on', col))
    inverter_7333.append(invlist)


meter_9068 = [
    ("9068_ac_power_data.csv",
     "measured_on",
     "meter_ac_power_(kw)_meter_150150"),
    ("9068_meter_data_20240101_20250430.csv",
     "measured_on",
     "meter_ac_power_(kw)_meter_150150")
]
#define inverter_9068 list
inverter_9068 = []
for inverter in ["inverter_module_1.1_ac_power_(kw)_inv_150135", "inverter_module_1.2_ac_power_(kw)_inv_150136", "inverter_module_1.3_ac_power_(kw)_inv_150137", "inverter_module_1.4_ac_power_(kw)_inv_150138", "inverter_module_2.1_ac_power_(kw)_inv_150139", "inverter_module_2.2_ac_power_(kw)_inv_150140", "inverter_module_2.3_ac_power_(kw)_inv_150141", "inverter_module_2.4_ac_power_(kw)_inv_150142",	"inverter_1_ac_power_(kw)_inv_150143", 	"inverter_2_ac_power_(kw)_inv_150144"]:
    #inverter is a column name
    #note that to get complete information, the data in the two invlists will have to be combined
    invlist = [('9068_ac_power_data.csv', "measured_on", inverter), ('9068_ac_power_data_20240101_20250430.csv', "measured_on", inverter)]
    inverter_9068.append(invlist)



#finally, define the dictionary!
systems = {
    2105: (meter_2105, inverter_2105),
    2107: (meter_2107, inverter_2107),
    7333: (meter_7333, inverter_7333),
    9068: (meter_9068, inverter_9068)
}

Will now iterate through the 8 lists stored in the values of the dictionary "systems". the meters and inverters will be done separately.

# Meters

In [ ]:
#I already ran this once and all files are created. I do not want to run it again. Run time ~1sec.

# for key in systems.keys():
#     system = Clean(key, '../../../../data_ds_project/systems/prize/')
#     data = systems[key][0] # list of tuples
#     cleaned = system.combine_and_clean(data)
#     daily = system.daily_average(cleaned)
#     system.write_to_file(daily,"meter")

# Inverters

We will need to account for the fact that any given system may have more than one inverter. Since systems often have inverters with rather unweildly names (and these names are not standardized with respect to other systems), we will rename the inverters and also create a new (csv) file that keeps track of systems, old inverter names, and new inverter names.

In [ ]:
#takes appx 1 hour to run

inverter_renaming = pd.DataFrame(columns = ['system_id', 'old_name', 'new_name'])

for key in systems.keys():
    system = Clean(key, '../../../../data_ds_project/systems/prize/')
    inverters = systems[key][1] # list of lists
    inv_num_new = 0
    for inverter in inverters:
        #inverter is a list of tuples
        data = inverter
        cleaned = system.combine_and_clean(data)
        daily = system.daily_average(cleaned)
        system.write_to_file(daily,"inverter",inv_num_new)

        #now must add to the dataframe of information
        new_row = [key, data[0][2], inv_num_new]
        inverter_renaming = pd.concat([inverter_renaming, pd.DataFrame([new_row], columns = inverter_renaming.columns)], ignore_index = True)

        inv_num_new +=1



In [ ]:
inverter_names = inverter_renaming.copy()
inverter_names['new_name'] = inverter_names['new_name'].apply(lambda x : str(x).zfill(3))
inverter_names

,system_id,old_name,new_name
0,2107,inv_01_ac_power_inv_149583,000
1,2107,inv_02_ac_power_inv_149588,001
2,2107,inv_03_ac_power_inv_149593,002
3,2107,inv_04_ac_power_inv_149598,003
4,2107,inv_05_ac_power_inv_149603,004
...,...,...,...
141,9068,inverter_module_2.2_ac_power_(kw)_inv_150140,005
142,9068,inverter_module_2.3_ac_power_(kw)_inv_150141,006
143,9068,inverter_module_2.4_ac_power_(kw)_inv_150142,007
144,9068,inverter_1_ac_power_(kw)_inv_150143,008


Since we renamed the inverters, we will save the information about the renaming (aka the above dataframe) to a csv file. It will be saved one level UP from the data files (so we can iterate through them without hitting this new file).

In [12]:
inverter_names.to_csv('../../../../data_ds_project/systems/prize/new_inverter_names_for_prize_cleaned_power.csv', index = False)